In [2]:
import os
os.chdir("/content/drive/MyDrive/kisco/Data")

In [3]:
import pandas as pd
import numpy as np

csv_data = pd.read_csv("wine.csv")
csv_data.head()

,alcohol,sugar,pH,class
0,9.4,1.9,3.51,0.0
1,9.8,2.6,3.20,0.0
2,9.8,2.3,3.26,0.0
3,9.8,1.9,3.16,0.0
4,9.4,1.9,3.51,0.0


In [4]:
wine_data = csv_data[["alcohol", "sugar", "pH"]]
wine_target = csv_data["class"]

In [9]:
from sklearn.model_selection import train_test_split

train_data, test_data, train_target, test_target = train_test_split(wine_data, wine_target, test_size=0.2, random_state=42)

sub_data, val_data, sub_target, val_target = train_test_split(train_data, train_target, test_size=0.2, random_state=42)

In [10]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(sub_data, sub_target)
print(dt_model.score(sub_data, sub_target))
print(dt_model.score(val_data, val_target))

0.9971133028626413
0.864423076923077


### 교차검증
![교차검증](https://img1.daumcdn.net/thumb/R1280x0/?scode=mtistory2&fname=https%3A%2F%2Fblog.kakaocdn.net%2Fdn%2FriP4b%2FbtqZlQyFueH%2FOJ1lk0AyTpEWlVMEbS0L01%2Fimg.png)

k개로 나눈 데이터셋의 부분집합들(한 칸씩)을 1번 fold, 2번 fold 라 부른다.

교차검증은 바로 그 fold 들을 차례대로 교차해 테스트 데이터로 사용한다는 말.

위 그림 예시에서는 첫 번째 모델 학습에서는 5번 fold를 테스트 데이터로 사용한다고 것이고, 다음 모델 학습에서는 4번 fold가 테스트 데이터이고, 그다음엔 3번 fold 이런 식으로 이어지면서 마지막 다섯 번째 모델 학습까지 차례대로 fold가 테스트 데이터로 교차되어 사용됨.

In [11]:
from sklearn.model_selection import cross_validate
scores = cross_validate(dt_model, train_data, train_target)
print(scores)

{'fit_time': array([0.01175189, 0.00720334, 0.00719666, 0.0068531 , 0.00709414]), 'score_time': array([0.00199103, 0.00180387, 0.00158143, 0.00150275, 0.00151634]), 'test_score': array([0.86923077, 0.84615385, 0.87680462, 0.84889317, 0.83541867])}


In [12]:
print(np.mean(scores["test_score"]))

0.855300214703487


## cross_validate : spliter
* 기본 5개의 폴더로 분리 
* 10개의 분리 : cv = 10
* cv=스플리트 객체
* cv=KFold() : 회귀모델
* cv=StratifiedKFold() : 분류모델

In [13]:
from sklearn.model_selection import StratifiedKFold
scores = cross_validate(dt_model, train_data, train_target, cv = StratifiedKFold())
print(np.mean(scores["test_score"]))

0.855300214703487


In [14]:
splitter = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_validate(dt_model, train_data, train_target, cv = splitter)
print(np.mean(scores["test_score"]))

0.8574181117533719


## min_impurity_decrease
* 부모노드와 자식노드의 불순도 차이 : 정보이득
* 불순도의 차이가 클수록 좋음
* 주어진 불순도의 최소값보다 작으면 가지분할(노드분할) 하지 않음

### n_jobs=n : n개의 코어를 사용하여 훈련
### n이 -1이면 이용가능한 모든코어 활용

In [20]:
from sklearn.model_selection import GridSearchCV
params = {"min_impurity_decrease":[0.0001, 0.0002, 0.0003, 0.0004, 0.0005]}

gs_model = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs = -1)
gs_model.fit(train_data, train_target)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'min_impurity_decrease': [0.0001, 0.0002, 0.0003,
                                                   0.0004, 0.0005]})

In [21]:
dt = gs_model.best_estimator_
print(dt.score(train_data, train_target))

0.9615162593804117


In [22]:
print(gs_model.best_params_)

{'min_impurity_decrease': 0.0001}


In [23]:
print(gs_model.cv_results_["mean_test_score"])

[0.86819297 0.86453617 0.86492226 0.86780891 0.86761605]


In [25]:
best_index = np.argmax(gs_model.cv_results_["mean_test_score"])
print(gs_model.cv_results_["params"][best_index])

{'min_impurity_decrease': 0.0001}


In [26]:
params = {'min_impurity_decrease': np.arange(0.0001, 0.001, 0.0001),
          'max_depth': range(5, 20, 1),
          'min_samples_split': range(2, 100, 10)
          }

In [28]:
gs_model = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs = -1)
gs_model.fit(train_data, train_target)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': range(5, 20),
                         'min_impurity_decrease': array([0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.0006, 0.0007, 0.0008,
       0.0009]),
                         'min_samples_split': range(2, 100, 10)})

In [29]:
print(gs_model.best_params_)

{'max_depth': 14, 'min_impurity_decrease': 0.0004, 'min_samples_split': 12}


In [30]:
print(np.max(gs_model.cv_results_["mean_test_score"]))
print(gs_model.cv_results_["mean_test_score"].shape)

0.8683865773302731
(1350,)


In [32]:
from scipy.stats import uniform, randint

rgen = randint(0, 10)
rgen.rvs(10)

array([2, 6, 5, 5, 5, 7, 8, 4, 6, 9])

In [ ]:
np.unique(rgen.rvs(1000), return_counts=True)

(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]),
 array([106, 106, 110, 103,  92,  90,  91,  91, 105, 106]))

In [ ]:
ugen = uniform(0, 1)
ugen.rvs(10)

array([0.59527773, 0.00314433, 0.85974505, 0.12314847, 0.24706896,
       0.76416696, 0.13779068, 0.31465355, 0.61029581, 0.94935477])

In [33]:
params = {'min_impurity_decrease': uniform(0.0001, 0.001),
          'max_depth': randint(20, 50),
          'min_samples_split': randint(2, 25),
          'min_samples_leaf': randint(1, 25),
          }

In [36]:
from sklearn.model_selection import RandomizedSearchCV

rs_model = RandomizedSearchCV(DecisionTreeClassifier(random_state=42), params, 
                        n_iter=100, n_jobs=-1, random_state=42)
rs_model.fit(train_data, train_target)

RandomizedSearchCV(estimator=DecisionTreeClassifier(random_state=42),
                   n_iter=100, n_jobs=-1,
                   param_distributions={'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7f8109baed10>,
                                        'min_impurity_decrease': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7f8109bae860>,
                                        'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7f8109baf220>,
                                        'min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7f8109baead0>},
                   random_state=42)

In [37]:
print(rs_model.best_params_)

{'max_depth': 39, 'min_impurity_decrease': 0.00034102546602601173, 'min_samples_leaf': 7, 'min_samples_split': 13}


In [38]:
print(np.max(rs_model.cv_results_["mean_test_score"]))
print(rs_model.cv_results_["mean_test_score"].shape)

0.8695428296438884
(100,)


In [39]:
dt =rs_model.best_estimator_
print(dt.score(test_data, test_target))

0.86
